# SkyGuard — Dataset Validation & Exploratory Analysis

This notebook validates the multi-station weather dataset used for the SkyGuard anomaly detection system.

## Objectives

1. Verify dataset integrity and completeness
2. Inspect temporal coverage
3. Analyze weather variable distributions
4. Verify meaningful variation between stations
5. Analyze spatial correlations
6. Quantify normal spatial variation
7. Validate whether the dataset is suitable for:
   - Statistical anomaly detection
   - Isolation Forest training
   - Spatial consistency analysis

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ncr_weather_network.parquet"

df = pd.read_parquet(DATA_PATH)

print(f"Dataset shape: {df.shape}")
display(df.head())

In [ ]:
print("Dataset Information")
print("=" * 50)

print(f"\nRows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Stations: {df['station_id'].nunique()}")

print("\nTemporal Coverage:")
print(f"Start: {df['timestamp'].min()}")
print(f"End:   {df['timestamp'].max()}")

print("\nData Types:")
display(df.dtypes)

print("\nMissing Values:")
display(df.isnull().sum())

In [ ]:
stations = (
    df[["station_id", "station_name", "latitude", "longitude"]]
    .drop_duplicates()
    .sort_values("station_id")
)

display(stations)

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(stations["longitude"], stations["latitude"], s=100)

for _, station in stations.iterrows():
    plt.annotate(
        station["station_name"],
        (station["longitude"], station["latitude"]),
        xytext=(5, 5),
        textcoords="offset points",
    )
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("SkyGuard Virtual Weather Station Network")
plt.grid(True)
plt.show()

In [ ]:
weather_columns = ["temperature", "humidity", "pressure", "surface_pressure"]
display(df[weather_columns].describe().T)

In [ ]:
sample_station = "AWS_001"
station_df = df[df["station_id"] == sample_station].copy()
station_df = station_df.set_index("timestamp")
daily = station_df[["temperature", "humidity", "pressure"]].resample("D").mean()

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(daily.index, daily["temperature"])
plt.title("Daily Mean Temperature - AWS_001")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(daily.index, daily["humidity"])
plt.title("Daily Mean Humidity - AWS_001")
plt.xlabel("Date")
plt.ylabel("Relative Humidity (%)")
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(daily.index, daily["pressure"])
plt.title("Daily Mean Pressure - AWS_001")
plt.xlabel("Date")
plt.ylabel("Pressure (hPa)")
plt.grid(True)
plt.show()

In [ ]:
temperature_wide = df.pivot(
    index="timestamp", columns="station_id", values="temperature"
)
display(temperature_wide.head())

In [ ]:
temp_corr = temperature_wide.corr()
display(temp_corr.round(4))

In [ ]:
variables = ["temperature", "humidity", "pressure"]

for variable in variables:
    wide = df.pivot(index="timestamp", columns="station_id", values=variable)
    corr = wide.corr()
    # Ignore diagonal when calculating range
    np.fill_diagonal(corr.values, np.nan)

    print(f"\n{variable.upper()} CORRELATION")
    print(f"Min: {np.nanmin(corr.values):.4f}")
    print(f"Max: {np.nanmax(corr.values):.4f}")
    print(f"Mean: {np.nanmean(corr.values):.4f}")

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(temp_corr, interpolation="nearest")
plt.colorbar()
plt.xticks(range(len(temp_corr.columns)), temp_corr.columns, rotation=45)
plt.yticks(range(len(temp_corr.index)), temp_corr.index)
plt.title("Temperature Correlation Between Stations")
plt.tight_layout()
plt.show()

In [ ]:
variables = ["temperature", "humidity", "pressure"]
for variable in variables:
    print(f"\n{'=' * 60}")
    print(variable.upper())
    print("=" * 60)
    wide = df.pivot(index="timestamp", columns="station_id", values=variable)
    identical_pairs = []
    station_ids = wide.columns.tolist()
    for i in range(len(station_ids)):
        for j in range(i + 1, len(station_ids)):
            a = station_ids[i]
            b = station_ids[j]
            if wide[a].equals(wide[b]):
                identical_pairs.append((a, b))
    if identical_pairs:
        print("Identical station pairs:")
        for pair in identical_pairs:
            print(pair)
    else:
        print("No identical station time series found.")

In [ ]:
def calculate_spatial_spread(df, variable):
    wide = df.pivot(index="timestamp", columns="station_id", values=variable)
    spread = pd.DataFrame(
        {
            "min": wide.min(axis=1),
            "max": wide.max(axis=1),
            "mean": wide.mean(axis=1),
            "std": wide.std(axis=1),
        }
    )
    spread["range"] = spread["max"] - spread["min"]
    return spread

In [ ]:
temp_spread = calculate_spatial_spread(df, "temperature")
humidity_spread = calculate_spatial_spread(df, "humidity")
pressure_spread = calculate_spatial_spread(df, "pressure")

In [ ]:
for name, spread in [
    ("Temperature", temp_spread),
    ("Humidity", humidity_spread),
    ("Pressure", pressure_spread),
]:
    print(f"\n{name} Spatial Spread")
    print("=" * 40)
    print(spread["range"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(temp_spread["range"], bins=100)
plt.title("Distribution of Temperature Spatial Range")
plt.xlabel("Max Temperature - Min Temperature (°C)")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(humidity_spread["range"], bins=100)
plt.title("Distribution of Humidity Spatial Range")
plt.xlabel("Max Humidity - Min Humidity (%)")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(pressure_spread["range"], bins=100)
plt.title("Distribution of Pressure Spatial Range")
plt.xlabel("Max Pressure - Min Pressure (hPa)")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
sample_time = pd.Timestamp("2025-06-15 12:00:00+00:00")
snapshot = df[df["timestamp"] == sample_time]
display(snapshot[["station_id", "station_name", "temperature", "humidity", "pressure"]])